# 🏃 MileRunner — run it free on Google Colab

This notebook installs MileRunner, starts the **autonomous trainer** in the background, and opens the **live dashboard** right here in the notebook. It's the quickest free way to watch AI agents learn to run a mile.

**How to use:** Runtime → *Run all* (or run each cell top to bottom). The dashboard appears in the last cell and refreshes every few seconds.

> Colab sessions are temporary (they disconnect after a while), so this is for exploring/demoing — for long-term training use a hosted container (see `docs/DEPLOY.md`).

## 1・ Install MileRunner and its dependencies

In [ ]:
!git clone -q https://github.com/granthicks14/claudeDRLsimulation.git
%cd claudeDRLsimulation
# torch is already installed in Colab; add the rest of the stack.
!pip install -q mujoco gymnasium 'stable-baselines3>=2.0' sb3-contrib dash plotly rich pyyaml tqdm
print('\n✅ Install complete.')

## 2・ Start the autonomous trainer in the background
It keeps training for as long as the notebook runs — the longer it goes, the faster the discovered mile.

In [ ]:
import subprocess, os, time
os.makedirs('experiments', exist_ok=True)
logf = open('experiments/train.log', 'w')
# 'hosted' is a small, fast config; raise population.size / timesteps_per_gen for stronger results.
trainer = subprocess.Popen(
    ['python', 'scripts/run.py', '--config', 'hosted'],
    stdout=logf, stderr=subprocess.STDOUT)
print('Trainer started (pid %d). Waiting for the first generation…' % trainer.pid)
for _ in range(60):
    if os.path.exists('experiments/status.json'):
        print('✅ First generation done — dashboard has data.'); break
    time.sleep(3)
else:
    print('Still on generation 0; the dashboard will fill in shortly.')

## 3・ Open the live dashboard
Speed / heart-rate / cadence / oxygen / energy / fatigue curves, per-algorithm comparison, and the animated 3D runner — all updating live.

In [ ]:
from milerunner.dashboard.app import create_app
app = create_app(db_path='experiments/milerunner.db',
                 status_path='experiments/status.json',
                 experiment='hosted')
# Dash renders inline inside Colab.
app.run(jupyter_mode='inline', port=8050)

## 4・ (optional) Peek at the best agent / research report
Run these anytime while training continues.

In [ ]:
!python scripts/analyze.py --experiment hosted --out experiments/report.md && echo '--- report ---' && cat experiments/report.md